In [1]:
# W8D1 - Cell 1
# Install packages required for the Production RAG API

!pip -q install fastapi uvicorn scikit-learn pandas numpy httpx pytest

In [2]:
# W8D1 - Cell 2
# Create the project structure

import os

base_dir = "/content/W8D1"

folders = [
    base_dir,
    f"{base_dir}/app",
    f"{base_dir}/tests",
    f"{base_dir}/results",
    f"{base_dir}/.github",
    f"{base_dir}/.github/workflows"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("W8D1 project folders created successfully.")
print()
for folder in folders:
    print(folder)

W8D1 project folders created successfully.

/content/W8D1
/content/W8D1/app
/content/W8D1/tests
/content/W8D1/results
/content/W8D1/.github
/content/W8D1/.github/workflows


In [3]:
# W8D1 - Cell 3
# Create a small knowledge base for the RAG system

documents = [
    {
        "id": "doc1",
        "text": "Retrieval Augmented Generation (RAG) combines document retrieval with language model generation. It retrieves relevant information before producing an answer."
    },
    {
        "id": "doc2",
        "text": "A vector database stores embeddings and supports similarity search. It is commonly used to retrieve relevant documents in RAG applications."
    },
    {
        "id": "doc3",
        "text": "FastAPI is a Python framework for building APIs. It provides automatic request validation and interactive API documentation."
    },
    {
        "id": "doc4",
        "text": "Docker packages an application and its dependencies into a container. Containers provide a consistent environment for running applications."
    },
    {
        "id": "doc5",
        "text": "CI/CD automates software integration, testing, building, and deployment. Automated pipelines help detect problems before deployment."
    },
    {
        "id": "doc6",
        "text": "RAG evaluation can measure retrieval quality, answer relevance, faithfulness, latency, and error rates."
    },
    {
        "id": "doc7",
        "text": "Monitoring a production AI application should track latency, request volume, errors, resource usage, and model or retrieval quality."
    },
    {
        "id": "doc8",
        "text": "RAG systems can fail because of poor document quality, incorrect retrieval, outdated information, weak chunking, or incorrect generated responses."
    }
]

print(f"Knowledge base contains {len(documents)} documents.\n")

for doc in documents:
    print(f"{doc['id']}: {doc['text']}")

Knowledge base contains 8 documents.

doc1: Retrieval Augmented Generation (RAG) combines document retrieval with language model generation. It retrieves relevant information before producing an answer.
doc2: A vector database stores embeddings and supports similarity search. It is commonly used to retrieve relevant documents in RAG applications.
doc3: FastAPI is a Python framework for building APIs. It provides automatic request validation and interactive API documentation.
doc4: Docker packages an application and its dependencies into a container. Containers provide a consistent environment for running applications.
doc5: CI/CD automates software integration, testing, building, and deployment. Automated pipelines help detect problems before deployment.
doc6: RAG evaluation can measure retrieval quality, answer relevance, faithfulness, latency, and error rates.
doc7: Monitoring a production AI application should track latency, request volume, errors, resource usage, and model or retri

In [4]:
# W8D1 - Cell 4
# Build a TF-IDF based document retriever

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts = [doc["text"] for doc in documents]
doc_ids = [doc["id"] for doc in documents]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(texts)


def retrieve(query, top_k=3):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, document_vectors)[0]

    ranked_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for index in ranked_indices:
        results.append({
            "id": documents[index]["id"],
            "text": documents[index]["text"],
            "score": float(similarities[index])
        })

    return results


print("Retriever created successfully.")
print("Vector matrix shape:", document_vectors.shape)

Retriever created successfully.
Vector matrix shape: (8, 84)


In [5]:
# W8D1 - Cell 5
# Test the retriever with sample questions

query = "What is RAG and how does it work?"

results = retrieve(query, top_k=3)

print("QUESTION:")
print(query)

print("\nTOP RETRIEVED DOCUMENTS:")
for rank, result in enumerate(results, start=1):
    print(f"\nRank {rank}")
    print("Document:", result["id"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"])

QUESTION:
What is RAG and how does it work?

TOP RETRIEVED DOCUMENTS:

Rank 1
Document: doc6
Score: 0.2146
Text: RAG evaluation can measure retrieval quality, answer relevance, faithfulness, latency, and error rates.

Rank 2
Document: doc2
Score: 0.1772
Text: A vector database stores embeddings and supports similarity search. It is commonly used to retrieve relevant documents in RAG applications.

Rank 3
Document: doc1
Score: 0.1664
Text: Retrieval Augmented Generation (RAG) combines document retrieval with language model generation. It retrieves relevant information before producing an answer.


In [6]:
# W8D1 - Cell 6
# Create a simple retrieval + answer pipeline

def rag_query(question, top_k=3):
    retrieved_docs = retrieve(question, top_k=top_k)

    context = " ".join(
        [item["text"] for item in retrieved_docs]
    )

    answer = (
        "Based on the retrieved knowledge: "
        + context
    )

    return {
        "question": question,
        "answer": answer,
        "sources": [item["id"] for item in retrieved_docs]
    }


questions = [
    "What is RAG?",
    "What does a vector database do?",
    "What is FastAPI?",
    "Why is Docker used?",
    "What is CI/CD?"
]

for question in questions:
    result = rag_query(question, top_k=3)

    print("=" * 80)
    print("QUESTION:", result["question"])
    print("ANSWER:", result["answer"])
    print("SOURCES:", result["sources"])

QUESTION: What is RAG?
ANSWER: Based on the retrieved knowledge: RAG evaluation can measure retrieval quality, answer relevance, faithfulness, latency, and error rates. A vector database stores embeddings and supports similarity search. It is commonly used to retrieve relevant documents in RAG applications. Retrieval Augmented Generation (RAG) combines document retrieval with language model generation. It retrieves relevant information before producing an answer.
SOURCES: ['doc6', 'doc2', 'doc1']
QUESTION: What does a vector database do?
ANSWER: Based on the retrieved knowledge: A vector database stores embeddings and supports similarity search. It is commonly used to retrieve relevant documents in RAG applications. RAG systems can fail because of poor document quality, incorrect retrieval, outdated information, weak chunking, or incorrect generated responses. RAG evaluation can measure retrieval quality, answer relevance, faithfulness, latency, and error rates.
SOURCES: ['doc2', 'doc8

In [7]:
# W8D1 - Cell 7
# Evaluate retrieval using Hit@1, Hit@3 and MRR

import pandas as pd

evaluation_data = [
    ("What is RAG?", "doc1"),
    ("What does a vector database do?", "doc2"),
    ("What is FastAPI?", "doc3"),
    ("Why use Docker?", "doc4"),
    ("What is CI/CD?", "doc5"),
    ("How can RAG systems be evaluated?", "doc6"),
    ("What should be monitored in a production AI application?", "doc7"),
    ("Why can RAG systems fail?", "doc8")
]

evaluation_rows = []

for question, expected_doc in evaluation_data:

    retrieved = retrieve(question, top_k=3)
    retrieved_ids = [item["id"] for item in retrieved]

    hit_at_1 = int(expected_doc == retrieved_ids[0])
    hit_at_3 = int(expected_doc in retrieved_ids)

    if expected_doc in retrieved_ids:
        rank = retrieved_ids.index(expected_doc) + 1
        reciprocal_rank = 1 / rank
    else:
        reciprocal_rank = 0

    evaluation_rows.append({
        "question": question,
        "expected": expected_doc,
        "retrieved": ", ".join(retrieved_ids),
        "hit_at_1": hit_at_1,
        "hit_at_3": hit_at_3,
        "reciprocal_rank": reciprocal_rank
    })

evaluation_df = pd.DataFrame(evaluation_rows)

print(evaluation_df.to_string(index=False))

print("\nEvaluation Summary")
print("Hit@1 :", round(evaluation_df["hit_at_1"].mean(), 3))
print("Hit@3 :", round(evaluation_df["hit_at_3"].mean(), 3))
print("MRR   :", round(evaluation_df["reciprocal_rank"].mean(), 3))

evaluation_df.to_csv(
    "/content/W8D1/results/evaluation_results.csv",
    index=False
)

print("\nSaved: results/evaluation_results.csv")

                                                question expected        retrieved  hit_at_1  hit_at_3  reciprocal_rank
                                            What is RAG?     doc1 doc6, doc2, doc1         0         1         0.333333
                         What does a vector database do?     doc2 doc2, doc8, doc6         1         1         1.000000
                                        What is FastAPI?     doc3 doc3, doc8, doc6         1         1         1.000000
                                         Why use Docker?     doc4 doc4, doc8, doc6         1         1         1.000000
                                          What is CI/CD?     doc5 doc5, doc8, doc7         1         1         1.000000
                       How can RAG systems be evaluated?     doc6 doc8, doc6, doc2         0         1         0.500000
What should be monitored in a production AI application?     doc7 doc7, doc4, doc6         1         1         1.000000
                               Why can R

In [8]:
# W8D1 - Cell 8
# Compare different top-k retrieval settings

import time

optimization_rows = []

test_questions = [
    "What is RAG?",
    "What is FastAPI?",
    "What is Docker?",
    "What is CI/CD?",
    "How should an AI system be monitored?"
]

for top_k in [1, 3, 5]:

    latencies = []

    for question in test_questions:
        start = time.perf_counter()

        _ = retrieve(question, top_k=top_k)

        end = time.perf_counter()

        latencies.append((end - start) * 1000)

    optimization_rows.append({
        "top_k": top_k,
        "average_latency_ms": np.mean(latencies),
        "maximum_latency_ms": np.max(latencies)
    })

optimization_df = pd.DataFrame(optimization_rows)

print(optimization_df.to_string(index=False))

optimization_df.to_csv(
    "/content/W8D1/results/optimization_results.csv",
    index=False
)

print("\nSaved: results/optimization_results.csv")

 top_k  average_latency_ms  maximum_latency_ms
     1            4.555888            4.848238
     3            2.469967            4.144319
     5            1.906055            2.124530

Saved: results/optimization_results.csv


In [9]:
# W8D1 - Cell 9
# Create the FastAPI application file

%%writefile /content/W8D1/app/main.py

from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


documents = [
    {
        "id": "doc1",
        "text": "Retrieval Augmented Generation (RAG) combines document retrieval with language model generation. It retrieves relevant information before producing an answer."
    },
    {
        "id": "doc2",
        "text": "A vector database stores embeddings and supports similarity search. It is commonly used to retrieve relevant documents in RAG applications."
    },
    {
        "id": "doc3",
        "text": "FastAPI is a Python framework for building APIs. It provides automatic request validation and interactive API documentation."
    },
    {
        "id": "doc4",
        "text": "Docker packages an application and its dependencies into a container. Containers provide a consistent environment for running applications."
    },
    {
        "id": "doc5",
        "text": "CI/CD automates software integration, testing, building, and deployment. Automated pipelines help detect problems before deployment."
    },
    {
        "id": "doc6",
        "text": "RAG evaluation can measure retrieval quality, answer relevance, faithfulness, latency, and error rates."
    },
    {
        "id": "doc7",
        "text": "Monitoring a production AI application should track latency, request volume, errors, resource usage, and model or retrieval quality."
    },
    {
        "id": "doc8",
        "text": "RAG systems can fail because of poor document quality, incorrect retrieval, outdated information, weak chunking, or incorrect generated responses."
    }
]


texts = [doc["text"] for doc in documents]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(texts)


app = FastAPI(
    title="W8D1 Production RAG API",
    version="1.0.0"
)


class QueryRequest(BaseModel):
    question: str
    top_k: int = 3


class QueryResponse(BaseModel):
    question: str
    answer: str
    sources: List[str]


def retrieve(question: str, top_k: int = 3):
    query_vector = vectorizer.transform([question])

    similarities = cosine_similarity(
        query_vector,
        document_vectors
    )[0]

    top_k = max(1, min(top_k, len(documents)))

    indices = np.argsort(similarities)[::-1][:top_k]

    return [documents[index] for index in indices]


@app.get("/")
def root():
    return {
        "message": "W8D1 Production RAG API is running"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy"
    }


@app.post("/query", response_model=QueryResponse)
def query(request: QueryRequest):

    retrieved = retrieve(
        request.question,
        request.top_k
    )

    context = " ".join(
        document["text"]
        for document in retrieved
    )

    answer = (
        "Based on the retrieved knowledge: "
        + context
    )

    return {
        "question": request.question,
        "answer": answer,
        "sources": [
            document["id"]
            for document in retrieved
        ]
    }

Writing /content/W8D1/app/main.py


In [10]:
# W8D1 - Cell 10

%%writefile /content/W8D1/requirements.txt

fastapi
uvicorn
pydantic
scikit-learn
numpy
pytest
httpx

Writing /content/W8D1/requirements.txt


In [11]:
# W8D1 - Cell 11

%%writefile /content/W8D1/tests/test_api.py

from fastapi.testclient import TestClient
from app.main import app


client = TestClient(app)


def test_root():
    response = client.get("/")
    assert response.status_code == 200


def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


def test_query():
    response = client.post(
        "/query",
        json={
            "question": "What is RAG?",
            "top_k": 3
        }
    )

    assert response.status_code == 200

    data = response.json()

    assert "question" in data
    assert "answer" in data
    assert "sources" in data
    assert len(data["sources"]) > 0

Writing /content/W8D1/tests/test_api.py


In [12]:
# W8D1 - Cell 12
# Run the tests and save the real output

!cd /content/W8D1 && pytest -v | tee results/test_output.txt

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W8D1
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collecting ... collected 0 items / 1 error

==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_api.py ______________________
ImportError while importing test module '/content/W8D1/tests/test_api.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_api.py:3: in <module>
    from app.main import app
E   ModuleNotFoundError: No module named 'app'
=========================== short test summary info ======================

In [13]:
# W8D1 - Cell 13

from fastapi.testclient import TestClient
import sys

sys.path.insert(0, "/content/W8D1")

from app.main import app

client = TestClient(app)

print("GET /")
response = client.get("/")
print(response.status_code)
print(response.json())

print("\nGET /health")
response = client.get("/health")
print(response.status_code)
print(response.json())

print("\nPOST /query")
response = client.post(
    "/query",
    json={
        "question": "What is RAG?",
        "top_k": 3
    }
)

print(response.status_code)
print(response.json())

GET /
200
{'message': 'W8D1 Production RAG API is running'}

GET /health
200
{'status': 'healthy'}

POST /query
200
{'question': 'What is RAG?', 'answer': 'Based on the retrieved knowledge: RAG evaluation can measure retrieval quality, answer relevance, faithfulness, latency, and error rates. A vector database stores embeddings and supports similarity search. It is commonly used to retrieve relevant documents in RAG applications. Retrieval Augmented Generation (RAG) combines document retrieval with language model generation. It retrieves relevant information before producing an answer.', 'sources': ['doc6', 'doc2', 'doc1']}


In [14]:
# W8D1 - Cell 14

%%writefile /content/W8D1/Dockerfile

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app ./app

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing /content/W8D1/Dockerfile


In [15]:
# W8D1 - Cell 15

%%writefile /content/W8D1/.dockerignore

__pycache__
*.pyc
.pytest_cache
.git
.ipynb_checkpoints
.env

Writing /content/W8D1/.dockerignore


In [16]:
# W8D1 - Cell 16

%%writefile /content/W8D1/.github/workflows/ci.yml

name: W8D1 CI/CD

on:
  push:
    branches:
      - main
      - "feat/**"

  pull_request:
    branches:
      - main


jobs:

  lint:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Basic syntax check
        run: python -m compileall app tests


  test:
    runs-on: ubuntu-latest
    needs: lint

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run tests
        run: pytest -v


  build:
    runs-on: ubuntu-latest
    needs: test

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Build Docker image
        run: docker build -t w8d1-production-rag .

Writing /content/W8D1/.github/workflows/ci.yml


In [17]:
# W8D1 - Cell 17

%%writefile /content/W8D1/monitoring.md

# W8D1 Production RAG Monitoring Strategy

## 1. Availability

Monitor the `/health` endpoint to verify that the API is available.

## 2. Latency

Track API response latency and retrieval latency to identify slow requests.

## 3. Error Rate

Monitor HTTP 4xx and 5xx responses and application exceptions.

## 4. Retrieval Quality

Track retrieval metrics such as Hit@1, Hit@3 and Mean Reciprocal Rank (MRR).

## 5. RAG Quality

Evaluate answer relevance, faithfulness and context quality using a representative evaluation dataset.

## 6. Resource Usage

Monitor CPU and memory usage of the Docker container.

## 7. Logging

Store application events, request information, errors and latency measurements while avoiding sensitive information.

## 8. Alerts

Configure alerts for API downtime, increased error rates, high latency and degradation in retrieval quality.

Writing /content/W8D1/monitoring.md


In [18]:
# W8D1 - Cell 18

%%writefile /content/W8D1/README.md

# W8D1 - Production RAG API

This project demonstrates a production-oriented Retrieval Augmented Generation API with retrieval evaluation, optimization, automated testing, Docker containerization, CI/CD and monitoring.

## Features

- TF-IDF based document retrieval
- RAG query pipeline
- Retrieval evaluation
- Hit@1
- Hit@3
- Mean Reciprocal Rank
- Retrieval latency comparison
- FastAPI API
- Automated tests
- Docker containerization
- GitHub Actions CI/CD
- Production monitoring strategy

## API Endpoints

### GET /

Returns the API status.

### GET /health

Returns the health status.

### POST /query

Example input:

```json
{
  "question": "What is RAG?",
  "top_k": 3
}

Writing /content/W8D1/README.md


In [20]:
# W8D1 - Cell 19

import os

for root, dirs, files in os.walk("/content/W8D1"):
    level = root.replace("/content/W8D1", "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

W8D1/
    .dockerignore
    Dockerfile
    requirements.txt
    monitoring.md
    README.md
    .pytest_cache/
        CACHEDIR.TAG
        .gitignore
        README.md
        v/
            cache/
                nodeids
                lastfailed
    tests/
        test_api.py
        __pycache__/
            test_api.cpython-313-pytest-8.4.2.pyc
    results/
        test_output.txt
        evaluation_results.csv
        optimization_results.csv
    app/
        main.py
        __pycache__/
            main.cpython-313.pyc
    .github/
        workflows/
            ci.yml


In [21]:
# W8D1 - Cell 20

import shutil

zip_path = shutil.make_archive(
    "/content/W8D1_project",
    "zip",
    "/content/W8D1"
)

print("Created:", zip_path)

Created: /content/W8D1_project.zip


In [22]:
from google.colab import files

files.download("/content/W8D1_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>